What This Code Does — Short Explanation
This code builds a Transformer Decoder from scratch using PyTorch, piece by piece.

The Big Picture: A Transformer Decoder takes a target sequence (e.g., a sentence being translated) and generates the next token step by step — attending to both what it has already generated AND what the encoder understood from the input.

The 7 Parts, Simply:
1. scaled_dot_product_attention — The core math formula. It computes how much each token should "pay attention" to other tokens using Query, Key, Value matrices. A mask blocks future tokens so the model can't "cheat."
2. MultiHeadAttention — Runs the attention formula h times in parallel (multiple heads), each learning different relationships, then combines them. Think of it as looking at the sentence from multiple perspectives at once.
3. FeedForward — A simple two-layer neural network applied to each token position independently. It adds non-linearity and transforms the representations.
4. PositionalEncoding — Since Transformers have no built-in sense of order, this injects position information using sine/cosine waves so the model knows token 1 vs token 5 vs token 10.
5. TransformerDecoderBlock — One full decoder layer combining all three sub-steps in sequence: self-attention → cross-attention → FFN, with residual connections and LayerNorm after each.
6. TransformerDecoder — Stacks N decoder blocks on top of each other (like 6 or 12 layers), wraps them with an embedding layer at the start and a linear+softmax at the end to produce vocabulary probabilities.
7. GPTStyleDecoder — A simplified decoder-only version (like GPT) — removes the cross-attention entirely since there's no encoder. Just masked self-attention + FFN, repeated N times.

Key Difference Between the Two Models:
TransformerDecoderGPTStyleDecoderHas encoder input?✅ Yes (cross-attention)❌ NoUse caseTranslation, summarizationText generation (GPT)Attention layers per block32


In [1]:
"""
Transformer Decoder Architecture
=================================
Full implementation of the Transformer Decoder from scratch using PyTorch.
Reference: Vaswani et al. (2017) — "Attention Is All You Need"

Components implemented:
  1. Scaled Dot-Product Attention
  2. Multi-Head Attention (with optional causal mask for self-attention)
  3. Position-wise Feed-Forward Network
  4. Positional Encoding
  5. Single Decoder Block (Masked Self-Attn → Cross-Attn → FFN)
  6. Full Transformer Decoder (stack of N blocks)
  7. Decoder-Only variant (like GPT — no cross-attention)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ─────────────────────────────────────────────────────────────
# 1. Scaled Dot-Product Attention
# ─────────────────────────────────────────────────────────────
def scaled_dot_product_attention(
    Q: torch.Tensor,        # (batch, heads, seq_q, d_k)
    K: torch.Tensor,        # (batch, heads, seq_k, d_k)
    V: torch.Tensor,        # (batch, heads, seq_k, d_v)
    mask: torch.Tensor = None,  # (batch, 1, seq_q, seq_k) — optional
    dropout: nn.Dropout = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
    Returns: (context, attention_weights)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, Sq, Sk)

    if mask is not None:
        # mask=0 means "ignore this position" → set to -inf before softmax
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)   # (B, H, Sq, Sk)
    attn_weights = torch.nan_to_num(attn_weights)  # handle all-masked rows

    if dropout is not None:
        attn_weights = dropout(attn_weights)

    context = torch.matmul(attn_weights, V)    # (B, H, Sq, d_v)
    return context, attn_weights


# ─────────────────────────────────────────────────────────────
# 2. Multi-Head Attention
# ─────────────────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention with h parallel attention heads.

    MultiHead(Q,K,V) = Concat(head_1,...,head_h) * W_O
    where head_i = Attention(Q*W_Qi, K*W_Ki, V*W_Vi)
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads   # dimension per head

        # Projection matrices
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)  # output projection

        self.attn_dropout = nn.Dropout(dropout)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(batch, seq, d_model) → (batch, heads, seq, d_k)"""
        B, S, _ = x.size()
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def forward(
        self,
        query: torch.Tensor,    # (B, Sq, d_model)
        key: torch.Tensor,      # (B, Sk, d_model)
        value: torch.Tensor,    # (B, Sk, d_model)
        mask: torch.Tensor = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        Q = self.split_heads(self.W_Q(query))  # (B, H, Sq, d_k)
        K = self.split_heads(self.W_K(key))
        V = self.split_heads(self.W_V(value))

        context, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.attn_dropout
        )

        # Concatenate heads: (B, H, Sq, d_k) → (B, Sq, d_model)
        B, H, Sq, _ = context.size()
        context = context.transpose(1, 2).contiguous().view(B, Sq, self.d_model)

        output = self.W_O(context)  # (B, Sq, d_model)
        return output, attn_weights


# ─────────────────────────────────────────────────────────────
# 3. Position-wise Feed-Forward Network
# ─────────────────────────────────────────────────────────────
class FeedForward(nn.Module):
    """
    FFN(x) = GELU(xW1 + b1)W2 + b2
    Applied independently to each position.
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))


# ─────────────────────────────────────────────────────────────
# 4. Positional Encoding
# ─────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (fixed, not learned).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ─────────────────────────────────────────────────────────────
# 5. Single Transformer Decoder Block
# ─────────────────────────────────────────────────────────────
class TransformerDecoderBlock(nn.Module):
    """
    One Transformer Decoder layer with three sub-layers:
      (a) Masked Multi-Head Self-Attention  (causal)
      (b) Multi-Head Cross-Attention        (to encoder output)
      (c) Position-wise Feed-Forward Network

    Each sub-layer wrapped with: LayerNorm(x + Sublayer(x))
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()

        # Sub-layer (a): Masked self-attention
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)

        # Sub-layer (b): Cross-attention (encoder–decoder attention)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)

        # Sub-layer (c): Feed-forward network
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,               # decoder input  (B, Tgt, d_model)
        encoder_output: torch.Tensor,   # encoder output (B, Src, d_model)
        src_mask: torch.Tensor = None,  # padding mask for encoder side
        tgt_mask: torch.Tensor = None,  # causal + padding mask for decoder side
    ) -> torch.Tensor:

        # (a) Masked self-attention
        _x, _ = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(_x))

        # (b) Cross-attention (Q from decoder, K/V from encoder)
        _x, _ = self.cross_attn(x, encoder_output, encoder_output, mask=src_mask)
        x = self.norm2(x + self.dropout(_x))

        # (c) Feed-forward network
        _x = self.ffn(x)
        x = self.norm3(x + self.dropout(_x))

        return x


# ─────────────────────────────────────────────────────────────
# 6. Full Transformer Decoder (N stacked blocks)
# ─────────────────────────────────────────────────────────────
class TransformerDecoder(nn.Module):
    """
    Full Transformer Decoder:
      Embedding → Positional Encoding → N Decoder Blocks → Linear → Softmax
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 512,
        num_heads: int = 8,
        d_ff: int = 2048,
        num_layers: int = 6,
        max_len: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        """Xavier/Glorot initialization for linear layers."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """
        Build look-ahead (causal) mask of shape (1, 1, seq_len, seq_len).
        Position i can only attend to positions 0..i.
        """
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, S, S)

    def forward(
        self,
        tgt: torch.Tensor,              # (B, Tgt) — token IDs
        encoder_output: torch.Tensor,   # (B, Src, d_model)
        src_mask: torch.Tensor = None,  # (B, 1, 1, Src) padding mask
        tgt_padding_mask: torch.Tensor = None,  # (B, 1, 1, Tgt) padding mask
    ) -> torch.Tensor:

        B, T = tgt.size()

        # Causal mask for decoder self-attention
        causal_mask = self.make_causal_mask(T, tgt.device)  # (1, 1, T, T)

        # Combine causal mask with padding mask
        if tgt_padding_mask is not None:
            tgt_mask = causal_mask & tgt_padding_mask  # broadcast over batch
        else:
            tgt_mask = causal_mask

        # Embedding + positional encoding
        x = self.pos_encoding(self.embedding(tgt) * math.sqrt(self.d_model))

        # Pass through N decoder blocks
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask=src_mask, tgt_mask=tgt_mask)

        x = self.norm(x)
        logits = self.fc_out(x)  # (B, T, vocab_size)
        return logits


# ─────────────────────────────────────────────────────────────
# 7. Decoder-Only Model (GPT-style — no cross-attention)
# ─────────────────────────────────────────────────────────────
class DecoderOnlyBlock(nn.Module):
    """Single block for a decoder-only (GPT-style) model."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        _x, _ = self.self_attn(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout(_x))
        _x = self.ffn(x)
        x = self.norm2(x + self.dropout(_x))
        return x


class GPTStyleDecoder(nn.Module):
    """Decoder-only Transformer (like GPT): no encoder, no cross-attention."""

    def __init__(self, vocab_size: int, d_model: int = 512, num_heads: int = 8,
                 d_ff: int = 2048, num_layers: int = 6, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            DecoderOnlyBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def make_causal_mask(self, seq_len, device):
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0).unsqueeze(0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        mask = self.make_causal_mask(T, x.device)
        x = self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model))
        for layer in self.layers:
            x = layer(x, mask=mask)
        return self.fc_out(self.norm(x))


# ─────────────────────────────────────────────────────────────
# Quick Smoke Test
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    VOCAB = 10000
    BATCH = 2
    SRC_LEN = 16
    TGT_LEN = 12
    D_MODEL = 128

    print("=" * 55)
    print("  Transformer Decoder — Smoke Test")
    print("=" * 55)

    # --- Encoder-Decoder Transformer ---
    decoder = TransformerDecoder(
        vocab_size=VOCAB, d_model=D_MODEL, num_heads=4,
        d_ff=512, num_layers=2, max_len=64
    )

    encoder_out = torch.randn(BATCH, SRC_LEN, D_MODEL)   # fake encoder output
    tgt_tokens = torch.randint(1, VOCAB, (BATCH, TGT_LEN))

    logits = decoder(tgt_tokens, encoder_out)
    print(f"\n[Encoder-Decoder] Input  : tgt={tgt_tokens.shape}, enc={encoder_out.shape}")
    print(f"[Encoder-Decoder] Output : logits={logits.shape}")  # (2, 12, 10000)

    probs = F.softmax(logits, dim=-1)
    predicted = probs.argmax(dim=-1)
    print(f"[Encoder-Decoder] Predicted tokens shape: {predicted.shape}")

    # --- Decoder-Only (GPT-style) ---
    gpt = GPTStyleDecoder(
        vocab_size=VOCAB, d_model=D_MODEL, num_heads=4,
        d_ff=512, num_layers=2, max_len=64
    )
    tokens = torch.randint(1, VOCAB, (BATCH, TGT_LEN))
    gpt_logits = gpt(tokens)
    print(f"\n[Decoder-Only/GPT] Input : {tokens.shape}")
    print(f"[Decoder-Only/GPT] Output: {gpt_logits.shape}")    # (2, 12, 10000)

    total_params = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
    print(f"\nTotal trainable params (Enc-Dec Decoder): {total_params:,}")
    print("\nAll tests passed!")

  Transformer Decoder — Smoke Test

[Encoder-Decoder] Input  : tgt=torch.Size([2, 12]), enc=torch.Size([2, 16, 128])
[Encoder-Decoder] Output : logits=torch.Size([2, 12, 10000])
[Encoder-Decoder] Predicted tokens shape: torch.Size([2, 12])

[Decoder-Only/GPT] Input : torch.Size([2, 12])
[Decoder-Only/GPT] Output: torch.Size([2, 12, 10000])

Total trainable params (Enc-Dec Decoder): 3,099,408

All tests passed!
